# FIR, AR and spectral whitening with shared identification

將 standalone demo 整合為同一 calling style：PSD/ACF/spread 比較，以及 raw 和 prewhitened input 的 LMS 識別。prewhitening 必須同時作用於 x 與 d；spectral fit 使用整段資料，屬 offline illustration。

來源：`demo_whitening.py`。所有資料與結果留在記憶體，圖表使用共用深色風格。

執行順序：設定 → 參數與資料 → 模型與訓練 → 圖表與數值摘要。

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Signal generation and whitening

In [ ]:
from adsp.experiments import whitening as experiment

n_samples, n_trials, seed, pole = 4096, 24, 7, 0.92
signals, ar, spectral = experiment.prepare(n_samples, rng=np.random.default_rng(seed), pole=pole)
print("Fitted AR coefficients:", ar.ar_coef)
for name, x in signals.items():
  print(name, "16-tap covariance spread:", stats.eigenvalue_spread(stats.covariance(x, n_taps=16)))

## 3. Shared dark PSD, autocorrelation and time plots

In [ ]:
fig, grid = plot.axes(rows=3)
plot.spectrum(signals, ax=grid[0, 0])
plot.lines({
  name: stats.autocorrelation(x, max_lag=40)
  for name, x in signals.items()
}, ax=grid[1, 0], xlabel="Lag", ylabel="Normalized autocorrelation", title="Whitening diagnostics")
plot.lines({name: x[:150] for name, x in signals.items()}, ax=grid[2, 0], title="Whitened signal segments")
plot.show(fig)

## 4. Raw versus pair-whitened system identification

In [ ]:
make_data = partial(data.system_identification, n_samples, h=experiment.PLANT_TAPS, pole=pole, noise_var=1e-4,
                    burn_in=512)
models = {
  "Raw LMS": partial(af.LMS, filter_order=15, mu=0.004),
  "Pair-whitened LMS": partial(af.LMS, filter_order=15, mu=0.004),
}
results = training.compare(models, make_data, n_trials=n_trials, seed=seed,
                           transforms={"Pair-whitened LMS": experiment.whiten_pair})
fig, grid = plot.axes(rows=2)
plot.learning(results, ax=grid[0, 0], window=64, title="MSE in each model's own input domain")
plot.learning(results, quantity="msd", ax=grid[1, 0], window=64, title="Shared physical plant coefficient MSD")
plot.show(fig)
training.summary(results, tail=1000)
print("Pair whitening changes the noise spectrum; compare shared coefficient MSD for plant accuracy.")